In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from loguru import logger
from langgraph.graph import StateGraph, START, END
from typing import TypedDict,Literal

load_dotenv()

# 初始化模型
model = init_chat_model(
    "qwen3.7-plus-2026-05-26",
    model_provider="openai",
    temperature=0.5,
    max_tokens=1024,
    timeout=60,
    max_retries=3,
    base_url=os.getenv("DASHSCOPE_API_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    extra_body={
        "thinking": {
            "type": "disabled"
            }
    }
)

class OverAllState(TypedDict):
    topic: str
    poem: str
    joke: str
    content_type: str

def node_a(state: OverAllState) -> OverAllState:
    '''
    生成七言绝句
    '''
    poem = model.invoke([HumanMessage(content=f"写一首关于{state['topic']}的七言绝句")]).content

    return {
        "poem": poem
    }

def node_b(state: OverAllState) -> OverAllState:
    '''
    生成笑话
    '''
    joke = model.invoke([HumanMessage(content=f"写一个关于{state['topic']}的笑话")]).content

    return {
        "joke": joke
    }

def audit_node(state: OverAllState) -> OverAllState:
    '''
    审核节点
    '''
    logger.info(f"任务节点已经全部执行完毕，诗{'已生成' if state['poem'] else '未生成'}，笑话{'已生成' if state['joke'] else '未生成'}，词{'已生成' if state['ci_poem'] else '未生成'}")
    return state


builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)

builder.add_node("audit_node", audit_node, defer=True)

builder.add_edge(START, "node_a")
builder.add_edge(START, "node_b")
builder.add_edge(START, "audit_node")
builder.add_edge("node_a", END)
builder.add_edge("node_b", END)
builder.add_edge("audit_node", END)

graph = builder.compile()
result = graph.invoke({"topic": "猫咪", "content_type": "诗"})
print(result)

from IPython.display import display
display(graph)